# Notes: 
Run the notebook from analysis directory

## TODOs:
1. In function: extract_entanglement_data, return final_result dataframe of the best entanglement candidate
2. In the for k, v in res.items() loop, write the infom file for backtrack
3. Align with the native structure

In [ ]:
import numpy as np
import pandas as pd
from topoly import lasso_type, make_surface
from topoly.params import PrecisionSurface, DensitySurface, SurfacePlotFormat, test
from topoly import gln
import topoly
import os, subprocess
import mdtraj as md
import openmm as mm
import openmm.app as app
import openmm.unit as unit

In [ ]:
# Define protein specific:
native_pdb = 'P36008_clean.pdb'


# extract frame from traj_idx and frame_idx and backmap to all-atom structure
analysis_dir = os.getcwd()
traj_dir = os.path.dirname(os.getcwd())  # Goes up 1 level from analysis/ to root
visual_dir = os.path.join(analysis_dir, 'visualization')
os.makedirs(visual_dir, exist_ok=True)

# Print directories information for debugging
print(f"analysis_dir: {analysis_dir}")
print(f"traj_dir: {traj_dir}")
print(f"visual_dir: {visual_dir}")

# Print the current working directory
print(f"Current working directory: {os.getcwd()}")

In [ ]:
def get_representative_macrostate_structure(msm_results_file, QG_file):

    msm = np.load(msm_results_file, allow_pickle=True)
    macrostate_assignment = msm['macrostate_assignment']
    macrostate_assignment_sorted = msm['macrostate_assignment_sorted']
    cluster = msm['cluster_centers_original']
    dtrajs = msm['discrete_trajectories']
    QG = np.load(QG_file)

    total_frames = dtrajs.shape[0]*dtrajs.shape[1]
    microstate_prob = [np.sum(dtrajs==i)/total_frames for i in range(len(cluster))]    

    # Get unique macrostate labels
    unique_macrostates = np.unique(macrostate_assignment_sorted)

    # Dictionary to store results for each macrostate
    macrostate_analysis = {}

    for macrostate in unique_macrostates:
        # Find all microstates that belong to this macrostate
        microstate_indices = np.where(macrostate_assignment_sorted == macrostate)[0]
        
        # Get probabilities for these microstates
        microstate_probabilities = [microstate_prob[i] for i in microstate_indices]
        
        # Find the microstate with the highest probability
        max_prob_index = np.argmax(microstate_probabilities)
        max_prob_microstate = microstate_indices[max_prob_index]
        max_probability = microstate_probabilities[max_prob_index]
        
        # Find all frames in dtrajs that belong to the highest probability microstate
        frame_indices = np.where(dtrajs == max_prob_microstate)
        
        # Get Q and G values for these frames
        frame_Q_values = QG[frame_indices[0], frame_indices[1], 0]  # Assuming Q is first column
        frame_G_values = QG[frame_indices[0], frame_indices[1], 1]  # Assuming G is second column
        
        # Get the cluster center for this microstate
        cluster_center = cluster[max_prob_microstate]
        
        # Calculate distances from each frame to the cluster center using np.linalg.norm
        frame_positions = np.column_stack((frame_Q_values, frame_G_values))
        distances_to_center = np.linalg.norm(frame_positions - cluster_center, axis=1)
        
        # Find frame with minimum distance to cluster center
        min_distance_idx = np.argmin(distances_to_center)
        
        # Get the actual trajectory and frame indices
        best_traj_idx = frame_indices[0][min_distance_idx]
        best_frame_in_traj = frame_indices[1][min_distance_idx]
        
        # Store results
        macrostate_analysis[macrostate] = {
            'microstate_indices': microstate_indices,
            'microstate_probabilities': microstate_probabilities,
            'highest_prob_microstate': max_prob_microstate,
            'highest_probability': max_probability,
            'best_frame_traj_idx': best_traj_idx,
            'best_frame_in_traj': best_frame_in_traj,
            'best_frame_Q': frame_Q_values[min_distance_idx],
            'best_frame_G': frame_G_values[min_distance_idx],
            'distance_to_center': distances_to_center[min_distance_idx],
            'cluster_center_Q': cluster_center[0],
            'cluster_center_G': cluster_center[1]
        }

    # Print results
    for macrostate, data in macrostate_analysis.items():
        print(f"Macrostate {macrostate}:")
        print(f"  Number of microstates: {len(data['microstate_indices'])}")
        print(f"  Microstate indices: {data['microstate_indices']}")
        print(f"  Highest probability microstate: {data['highest_prob_microstate']}")
        print(f"  Highest probability: {data['highest_probability']:.6f}")
        print(f"  Cluster center: [{data['cluster_center_Q']:.6f}, {data['cluster_center_G']:.6f}]")
        print(f"  Best frame: Trajectory {data['best_frame_traj_idx']}, Frame {data['best_frame_in_traj']}")
        print(f"  Best frame Q: {data['best_frame_Q']:.6f}")
        print(f"  Best frame G: {data['best_frame_G']:.6f}")
        print(f"  Distance to center: {data['distance_to_center']:.6f}")
        print()

    return macrostate_analysis

def extract_frame_from_trajectory(traj_file, psf_file, index, output_pdb):
    """Load trajectory and extract specific frame"""
    try:
        t = md.load_dcd(traj_file, psf_file)
        t.center_coordinates()
        t[index].save(output_pdb, force_overwrite=True)
        return t
    except Exception as e:
        print(f"Error loading trajectory: {e}")
        exit(1)

def run_pulchra(input_pdb):
    """Run Pulchra on input PDB file"""
    output_pdb = input_pdb.replace('.pdb', '.rebuilt.pdb')
    try:
        result = subprocess.run(['pulchra', input_pdb], 
                              capture_output=True, text=True, check=True)
        print("Pulchra completed successfully")
        return output_pdb
    except subprocess.CalledProcessError as e:
        print(f"Pulchra failed: {e}")
        print(f"Error output: {e.stderr}")
        exit(1)
def minimize_structure(pdb_file, output_file, device='CPU'):
    """
    Perform energy minimization on structure using OpenMM.
    
    This function loads a PDB structure, adds hydrogens, creates a force field system,
    applies C-alpha restraints, and performs energy minimization. It prints the potential
    energy before and after minimization to monitor the optimization process.
    
    Args:
        pdb_file (str): Path to input PDB file (output from Pulchra)
        output_file (str): Path to save the minimized structure
        
    Returns:
        openmm.app.Simulation: The simulation object after minimization
        
    Note:
        The function prints:
        - Initial potential energy before minimization
        - Final potential energy after minimization  
        - Energy change (should be negative, indicating energy reduction)
        
        Uses AMBER14 force field with:
        - No cutoff for non-bonded interactions
        - Hydrogen bond constraints
        - C-alpha atom restraints (5.0 kcal/mol/Å²)
        - Langevin dynamics at 300K
        - Maximum 100 minimization iterations
    """
    # Define magic numbers to constants at the top
    RESTRAINT_FORCE_CONSTANT = 50.0 * unit.kilocalories_per_mole / unit.angstroms**2
    TEMPERATURE = 300 * unit.kelvin
    FRICTION = 1.0 / unit.picosecond
    TIMESTEP = 0.002 * unit.picoseconds
    MAX_MINIMIZATION_ITERATIONS = 100

    # Load Pulchra output
    pdb = mm.app.PDBFile(pdb_file)
    
    # Add hydrogens
    forcefield = mm.app.ForceField("amber14-all.xml")
    modeller = mm.app.Modeller(pdb.topology, pdb.positions)
    modeller.addHydrogens(forcefield)
    
    # Build system
    system = forcefield.createSystem(
        modeller.topology,
        nonbondedMethod=app.NoCutoff,
        constraints=app.HBonds
    )
    
    # Add restraints
    restraint_force = create_ca_restraints(modeller, RESTRAINT_FORCE_CONSTANT)
    system.addForce(restraint_force)
    
    # Setup simulation
    integrator = mm.LangevinIntegrator(TEMPERATURE, FRICTION, TIMESTEP)
    if device == 'CPU':
        platform = mm.Platform.getPlatformByName("CPU")
        properties = {'Threads': '4'}
    elif device == 'GPU':
        platform = mm.Platform.getPlatformByName("CUDA")
        properties = {'CudaPrecision': 'mixed', "DeviceIndex": "0"}
    else:
        raise ValueError(f"Invalid device: {device}")

    simulation = app.Simulation(modeller.topology, system, integrator, platform, platformProperties=properties)
    simulation.context.setPositions(modeller.positions)
    
    # Get initial potential energy
    initial_state = simulation.context.getState(getEnergy=True)
    initial_energy = initial_state.getPotentialEnergy()
    print(f"Initial potential energy: {initial_energy}")
    
    # Minimize
    print("Minimizing energy in vacuum...")
    simulation.minimizeEnergy(maxIterations=MAX_MINIMIZATION_ITERATIONS)
    
    # Get final potential energy
    final_state = simulation.context.getState(getEnergy=True)
    final_energy = final_state.getPotentialEnergy()
    print(f"Final potential energy: {final_energy}")
    print(f"Energy change: {final_energy - initial_energy}")
    
    # Save result
    positions = simulation.context.getState(getPositions=True).getPositions()
    with open(output_file, "w") as f:
        app.PDBFile.writeFile(modeller.topology, positions, f)
    
    return simulation

def create_ca_restraints(modeller, k_restraint):
    """Create C-alpha atom restraints"""
    restraint_force = mm.CustomExternalForce(
        "0.5*k*((x-x0)^2 + (y-y0)^2 + (z-z0)^2)"
    )
    restraint_force.addPerParticleParameter("k")
    restraint_force.addPerParticleParameter("x0")
    restraint_force.addPerParticleParameter("y0")
    restraint_force.addPerParticleParameter("z0")
    
    for atom in modeller.topology.atoms():
        if atom.name == "CA":
            pos = modeller.positions[atom.index]
            restraint_force.addParticle(atom.index, [k_restraint, pos[0], pos[1], pos[2]])
    
    return restraint_force


def get_crossing(i: int, j: int, pdb_file: str, threading_terminal: str) -> list[int]:
    """
    Run topoly lasso_type on a given contact (i, j).

    Parameters
    ----------
    i, j : int
        Residue indices forming the contact.
    pdb_file : str
        Path to the PDB structure file (must exist).
    threading_terminal : {'Nter', 'Cter'}
        Which terminal threading to extract crossings from.

    Returns
    -------
    crossing_residues : list[int]
        List of crossing residue indices (absolute values).
        Returns [] if no crossings are detected.
    """
    i, j = int(i), int(j)  # ensure plain Python ints
    res = lasso_type(
        pdb_file,
        [i, j],
        min_dist=[10, 4, 5],
        pic_files=0,  # SurfacePlotFormat.VMD if you want plots
        output_prefix='misfold',
        more_info=True
    )

    # Extract the inner dictionary
    _, value = next(iter(res.items()))

    if threading_terminal == 'Nter':
        crossings = value.get('crossingsN', [])
    elif threading_terminal == 'Cter':
        crossings = value.get('crossingsC', [])
    else:
        raise ValueError("threading_terminal must be 'Nter' or 'Cter'")

    return [abs(int(c)) for c in crossings]

def write_vmd_script(pdb_file, i, j, crossings=None, g_val=0, native_structure=None, output="render.tcl"):
    """
    Generate a VMD TCL script for visualizing protein entanglement.

    Parameters
    ----------
    pdb_file : str
        Path to the PDB structure file to load in VMD.
    i : int
        Residue index of the first contact residue.
    j : int
        Residue index of the second contact residue.
    crossings : list of int, optional
        Residue indices that form the threading (crossing residues).
        Each will be highlighted with ±6 residues in blue cartoon,
        and its CA atom in magenta VDW spheres.
    g_val : int, optional
        G value of the contact. If 0, no entanglement will be shown.
    native_structure : str, optional
        Path to the native structure PDB file for alignment.
        If provided, the pdb_file will be aligned to this structure.
    output : str, optional
        Output filename for the TCL script (default: "render.tcl").
    """

    # Ensure crossings is always a list
    crossings = crossings or []

    # Header - modified to handle native structure alignment
    if native_structure:
        tcl_header = f"""# --- AUTO-GENERATED VMD SCRIPT WITH ALIGNMENT ---
set pdb_file "{pdb_file}"
set native_structure "{native_structure}"

# Load native structure first (reference)
mol new $native_structure
mol delrep 0 top

# Create representation for native structure (color ID 7)
mol representation NewCartoon
mol color ColorID 7
mol selection "all"
mol material Edgy
mol addrep top

# Add entanglement representation to native structure
mol representation VDW 0.7 80
mol color ColorID 4
mol selection "(resid {i} or resid {j}) and name CA"
mol material Edgy
mol addrep top

# Add loop region representation to native structure (red)
mol representation NewCartoon
mol color ColorID 1
mol selection "resid {i} to {j}"
mol material Edgy
mol addrep top

# Add crossing residues representation to native structure
mol representation VDW 1.2 40
mol color ColorID 13
mol selection "(resid {' or resid '.join(str(c) for c in crossings)}) and name CA"
mol material Edgy
mol addrep top

# Add threading regions representation to native structure (blue, +/-6 around crossings)
"""
        # Add threading regions for each crossing
        for crossing in crossings:
            tcl_header += f"""
mol representation NewCartoon
mol color ColorID 0
mol selection "resid {max(crossing - 6, 1)} to {crossing + 6}"
mol material Edgy
mol addrep top
"""

        tcl_header += """
# Load structure to be aligned
mol new $pdb_file
mol delrep 0 top

# Align the second structure to the native structure using all CA atoms
set sel1 [atomselect 0 "name CA"]
set sel2 [atomselect 1 "name CA"]
set transformation_matrix [measure fit $sel2 $sel1]
$sel2 move $transformation_matrix

# Set the aligned structure as the active molecule
mol top 1
"""
    else:
        tcl_header = f"""# --- AUTO-GENERATED VMD SCRIPT ---
set pdb_file "{pdb_file}"
mol new $pdb_file
mol delrep 0 top

"""

    # Entanglement section (only if g_val != 0 and crossings exist)
    tcl_entanglement = """
# Contact residues
set i __I__
set j __J__

# Crossing residues (list)
set crossings { __CROSSINGS__ }

# Bond between contact residues (CA atoms)
set sel [atomselect top "(resid $i or resid $j) and name CA"]
set idx [$sel get index]
if {[llength $idx] == 2} {
    topo addbond [lindex $idx 0] [lindex $idx 1]
}

# Bonds representation for contacts
mol representation Bonds 0.5 12.0
mol color ColorID 4
mol selection "(resid $i or resid $j) and name CA"
mol material Edgy
mol addrep top

# VDW for contact residues
mol representation VDW 0.7 80
mol color ColorID 4
mol selection "(resid $i or resid $j) and name CA"
mol material Edgy
mol addrep top

# Loop region in red
mol selection "resid $i to $j"
mol representation NewCartoon
mol color ColorID 1
mol material Edgy
mol addrep top

# Threading residues and crossings
foreach id3 $crossings {
    set threading_start [expr {max($id3 - 6, 1)}]
    set threading_end [expr {$id3 + 6}]
    
    mol selection "resid $threading_start to $threading_end"
    mol representation NewCartoon
    mol color ColorID 0
    mol material Edgy
    mol addrep top

    # Highlight CA atom of crossing residue in magenta
    mol selection "resid $id3 and name CA"
    mol representation VDW 1.2 40
    mol color ColorID 13
    mol material Edgy
    mol addrep top
}
"""

    # Rest of protein (always included, with safe guard)
    tcl_rest_of_protein = """
# Rest of protein in white
set loop_range "resid $i to $j"

if {[llength $crossings] > 0} {
    set cross_ranges [join [lmap id3 $crossings {format "(resid %d to %d)" [expr {$id3 - 6}] [expr {$id3 + 6}]}] " or "]
    mol selection "not (($loop_range) or $cross_ranges)"
} else {
    mol selection "not ($loop_range)"
}
mol representation NewCartoon
mol color ColorID 8
mol material Edgy
mol addrep top

# Display settings
color Display Background white
color scale method BGR
axes location off
display projection orthographic
"""

    # Assemble script
    if g_val != 0 and crossings:
        tcl_template = tcl_header + tcl_entanglement + tcl_rest_of_protein
    else:
        tcl_template = tcl_header + tcl_rest_of_protein

    # Replace placeholders
    tcl_code = (
        tcl_template
        .replace("__I__", str(i))
        .replace("__J__", str(j))
        .replace("__CROSSINGS__", " ".join(str(c) for c in crossings))
    )

    # Write out TCL file
    with open(output, "w") as f:
        f.write(tcl_code)

    print(f"[INFO] Generated VMD script: {output}")

def extract_entanglement_data(traj_dir, traj_idx, frame_idx, mini_output, native_structure):
    """
    Extract entanglement data from trajectory files and determine crossing residues.
    
    Parameters
    ----------
    traj_dir : str
        Directory containing trajectory data
    traj_idx : str or int
        Trajectory index/identifier
    frame_idx : int
        Frame index to analyze
    mini_output : str
        Path to minimized PDB file
        
    Returns
    -------
    tuple
        (i_val, j_val, threading_type, g_val, crossings)
    """
    # Read entanglement info
    df = pd.read_csv(f"{traj_dir}/{traj_idx}/GQ/G/{traj_idx}.EntInfo")
    ref = df[df['Frame'] == -1]
    frame = df[df['Frame'] == frame_idx]
    final_result = select_best_entanglement(ref, frame)

    # Extract basic values
    i_val = int(final_result['i'].iloc[0] + 1)  # shift by 1 since residue indices start from 1 and python from 0
    j_val = int(final_result['j'].iloc[0] + 1)
    threading_type = final_result['origin'].iloc[0]

    # Check origin and set g_val accordingly
    if final_result['origin'].iloc[0] == 'Nter':
        g_val_frame = final_result['Gn_frame'].iloc[0]
        g_val_ref = final_result['Gn_ref'].iloc[0]
    else:  # Cter
        g_val_frame = final_result['Gc_frame'].iloc[0]
        g_val_ref = final_result['Gc_ref'].iloc[0]

    # Get crossing residues if entangled
    if g_val_frame != 0:
        crossings_frame = get_crossing(
            i=i_val,
            j=j_val,
            pdb_file=mini_output,
            threading_terminal=threading_type
        )
    else:
        crossings_frame = []
    print("Crossing residue in frame: ")
    print(crossings_frame)
    
    if g_val_ref != 0:
        crossings_ref = get_crossing(
            i=i_val,
            j=j_val,
            pdb_file=native_structure,
            threading_terminal=threading_type
        )
    else:
        crossings_ref = []
    
    print("Crossing in Ref Structure:")
    print(crossings_ref)
        
    # group crossing frame and crossing ref
    crossings = list(set(crossings_frame + crossings_ref))
    
    g_val = g_val_frame if abs(g_val_frame) > abs(g_val_ref) else g_val_ref
    
    return i_val, j_val, crossings, g_val, final_result


def select_best_entanglement(ref: pd.DataFrame, frame: pd.DataFrame) -> pd.DataFrame:
    """
    Compare entanglements between reference and frame DataFrames and 
    return the best candidate contact for visualization.

    The best contact is chosen by:
    1. Finding the contact with maximum difference in Gn (N-terminal entanglement).
       - If multiple, select the one with maximum gn_diff.
    2. Finding the contact with maximum difference in Gc (C-terminal entanglement).
       - If multiple, select the one with maximum gc_diff.
    3. From both candidates (Nter and Cter), choose the one with the smallest loop size.

    Parameters
    ----------
    ref : pd.DataFrame
        Reference DataFrame (Frame == -1).
    frame : pd.DataFrame
        Frame DataFrame (Frame != -1).

    Returns
    -------
    final_result : pd.DataFrame
        DataFrame with a single row corresponding to the best entanglement.
        Includes an additional column 'origin' = {'Nter', 'Cter'} to show which terminal.
    """
    # Keep only necessary columns
    keep_cols = ['Frame', 'i', 'j', 'gn', 'Gn', 'gc', 'Gc']
    ref = ref[keep_cols]
    frame = frame[keep_cols]

    # Merge reference and frame
    merged = pd.merge(ref, frame, on=['i', 'j'], how='inner', suffixes=('_ref', '_frame'))
    
    # Differences
    merged['Gn_diff'] = (merged['Gn_frame'] - merged['Gn_ref']).abs()
    merged['gn_diff'] = (merged['gn_frame'] - merged['gn_ref']).abs()
    merged['Gc_diff'] = (merged['Gc_frame'] - merged['Gc_ref']).abs()
    merged['gc_diff'] = (merged['gc_frame'] - merged['gc_ref']).abs()
    merged['loop_size'] = (merged['j'] - merged['i']).abs()

    # --- N-terminal candidate ---
    subset_Nter = merged[merged['Gn_diff'] == merged['Gn_diff'].max()]
    result_Nter = subset_Nter[subset_Nter['gn_diff'] == subset_Nter['gn_diff'].max()]
    best_Nter = result_Nter.loc[[result_Nter['loop_size'].idxmin()]].copy()
    best_Nter['origin'] = 'Nter'

    # --- C-terminal candidate ---
    subset_Cter = merged[merged['Gc_diff'] == merged['Gc_diff'].max()]
    result_Cter = subset_Cter[subset_Cter['gc_diff'] == subset_Cter['gc_diff'].max()]
    best_Cter = result_Cter.loc[[result_Cter['loop_size'].idxmin()]].copy()
    best_Cter['origin'] = 'Cter'

    # # --- Compare Nter vs Cter ---
    # candidates = pd.concat([best_Nter, best_Cter], ignore_index=True)
    # final_result = candidates.loc[[candidates['loop_size'].idxmin()]]
        # --- Compare Nter vs Cter using new criteria ---
        
    # Step 1: Compare absolute differences (Gn_diff vs Gc_diff)
    if best_Nter['Gn_diff'].iloc[0] > best_Cter['Gc_diff'].iloc[0]:
        final_result = best_Nter
    elif best_Nter['Gn_diff'].iloc[0] < best_Cter['Gc_diff'].iloc[0]:
        final_result = best_Cter
    else:
        # Step 2: If absolute differences are equal, compare gn_diff vs gc_diff
        # if best_Nter['gn_diff'].iloc[0] > best_Cter['gc_diff'].iloc[0]:
        if abs(best_Nter['gn_frame'].iloc[0]) > abs(best_Cter['gc_frame'].iloc[0]):
            final_result = best_Nter
        # elif best_Nter['gn_diff'].iloc[0] < best_Cter['gc_diff'].iloc[0]:
        elif abs(best_Nter['gn_frame'].iloc[0]) < abs(best_Cter['gc_frame'].iloc[0]):
            final_result = best_Cter
        else:
            # Step 3: If those are also equal, compare loop_size
            if best_Nter['loop_size'].iloc[0] <= best_Cter['loop_size'].iloc[0]:
                final_result = best_Nter
            else:
                final_result = best_Cter
    
    return final_result



In [ ]:
res = get_representative_macrostate_structure(f'{analysis_dir}/msm_results/msm_analysis_results.npz', 'QG.npy')

In [ ]:

for k, v in res.items():
    print(f"Processing State {k}...")
    os.chdir(visual_dir)
    traj_idx = v['best_frame_traj_idx']
    frame_idx = v['best_frame_in_traj']
    # print(k, traj_idx, frame_idx)
    extract_frame_from_trajectory(f"{traj_dir}/{traj_idx}/{traj_idx}_prod.dcd", f"{traj_dir}/template/setup/top.psf", frame_idx, 'temp.pdb')
    pulchra_output = run_pulchra('temp.pdb')
    # minimize the pulchra output structure
    mini_output = f'State{k}.minimized.pdb'
    simulation = minimize_structure(pulchra_output, mini_output, device='GPU')
    i_val, j_val, crossings, g_val, final_result = extract_entanglement_data(traj_dir, traj_idx, frame_idx, mini_output, f"{traj_dir}/template/setup/{native_pdb}")
    print(f" Representative structure for State {k}")
    print(f"Trajectory: {traj_idx}, frame: {frame_idx}")
    print("Entanglement information:")
    print(final_result)
    print(f"Representative entanglement: contact:[{i_val}, {j_val}], crossings: {crossings}, g_val_frame: {g_val}")
    write_vmd_script(mini_output, i_val, j_val, crossings, g_val, native_structure=f"{traj_dir}/template/setup/{native_pdb}", output=f"State{k}.tcl")
    print("---------------------------------------------")
